In [11]:
import rasterio
from rasterio.windows import Window
from PIL import Image
import numpy as np
import os

def geotiff_slicing(input_tiff_path, output_folder, output_size=(1024, 1024), to_jpg=False, slicing=True):
    """
    Slices a GeoTIFF image into smaller square tiles and optionally converts the slices to JPEG.
    
    Parameters:
    - input_tiff_path: str, Path to the input GeoTIFF image.
    - output_folder: str, Path to the folder where output images will be saved.
    - output_size: tuple, Desired (width, height) of the output slices (default is 1024x1024).
    - to_jpg: bool, If True, converts the output slices to JPG format (default is False).
    - slicing: bool, If True, slices the GeoTIFF image. If False, saves the entire image without slicing.

    Returns:
    - List of paths to the output images (either .tiff or .jpg).
    """

    # Open the GeoTIFF file
    with rasterio.open(input_tiff_path) as dataset:
        output_paths = []

        if slicing:
            # Get the width and height of the image
            img_width = dataset.width
            img_height = dataset.height
            tile_width, tile_height = output_size

            # Calculate the number of slices in each direction, ensuring full coverage of the image
            x_slices = (img_width + tile_width - 1) // tile_width  # Number of tiles along width (ceil division)
            y_slices = (img_height + tile_height - 1) // tile_height  # Number of tiles along height (ceil division)

            for i in range(y_slices):
                for j in range(x_slices):
                    # Calculate window coordinates
                    x_off = min(j * tile_width, img_width - tile_width)
                    y_off = min(i * tile_height, img_height - tile_height)

                    # Define the window for each slice
                    window = Window(x_off, y_off, tile_width, tile_height)

                    # Read the window from the GeoTIFF
                    transform = dataset.window_transform(window)
                    img_data = dataset.read(window=window)

                    # Create the output file name
                    output_filename = f"tile_{i}_{j}.tiff"
                    output_path = os.path.join(output_folder, output_filename)

                    # Write the slice to a new GeoTIFF file
                    with rasterio.open(
                        output_path, 'w',
                        driver='GTiff',
                        height=tile_height,
                        width=tile_width,
                        count=dataset.count,
                        dtype=img_data.dtype,
                        crs=dataset.crs,
                        transform=transform
                    ) as dest:
                        dest.write(img_data)

                    # Convert to JPEG if required
                    if to_jpg:
                        img = Image.fromarray(np.moveaxis(img_data, 0, -1))  # Convert to (H, W, C) format

                        # Check if the image is RGBA and convert it to RGB
                        if img.mode == 'RGBA':
                            img = img.convert('RGB')

                        output_jpg_path = output_path.replace('.tiff', '.jpg')
                        img.save(output_jpg_path, 'JPEG')
                        os.remove(output_path)  # Remove the .tiff if converting to .jpg
                        output_paths.append(output_jpg_path)
                    else:
                        output_paths.append(output_path)

        else:
            # Read the entire image
            img_data = dataset.read()
            output_filename = "full_image.tiff"
            output_path = os.path.join(output_folder, output_filename)

            # Write the entire image to a new GeoTIFF file
            with rasterio.open(
                output_path, 'w',
                driver='GTiff',
                height=dataset.height,
                width=dataset.width,
                count=dataset.count,
                dtype=img_data.dtype,
                crs=dataset.crs,
                transform=dataset.transform
            ) as dest:
                dest.write(img_data)

            # Convert to JPEG if required
            if to_jpg:
                img = Image.fromarray(np.moveaxis(img_data, 0, -1))  # Convert to (H, W, C) format

                # Check if the image is RGBA and convert it to RGB
                if img.mode == 'RGBA':
                    img = img.convert('RGB')

                output_jpg_path = output_path.replace('.tiff', '.jpg')
                img.save(output_jpg_path, 'JPEG')
                os.remove(output_path)  # Remove the .tiff if converting to .jpg
                output_paths.append(output_jpg_path)
            else:
                output_paths.append(output_path)

    return output_paths


In [12]:
input_tiff = "m_3008501_ne_16_030_20211103.tif"
output_dir = "output_tiles_2"
output_tile_size = (8192, 8192)  # Slice size: 512x512 pixels


# Slice and convert to JPG
output_paths = geotiff_slicing(input_tiff, output_dir, output_tile_size, to_jpg=True, slicing=True)

# Save entire image without slicing
# output_paths_full = geotiff_slicing(input_tiff, output_dir, to_jpg=True, slicing=False)

print("Generated slices:", output_paths)
# print("Full image:", output_paths_full)

Generated slices: ['output_tiles_2\\tile_0_0.jpg', 'output_tiles_2\\tile_0_1.jpg', 'output_tiles_2\\tile_0_2.jpg', 'output_tiles_2\\tile_1_0.jpg', 'output_tiles_2\\tile_1_1.jpg', 'output_tiles_2\\tile_1_2.jpg', 'output_tiles_2\\tile_2_0.jpg', 'output_tiles_2\\tile_2_1.jpg', 'output_tiles_2\\tile_2_2.jpg', 'output_tiles_2\\tile_3_0.jpg', 'output_tiles_2\\tile_3_1.jpg', 'output_tiles_2\\tile_3_2.jpg']


In [9]:
import math

max_pixels = 178956970
X_exact = math.sqrt(max_pixels)
X_power_of_2 = 2 ** math.floor(math.log2(X_exact))

print("Exact X:", X_exact)
print("Closest lower power of 2:", X_power_of_2)


Exact X: 13377.479957002364
Closest lower power of 2: 8192
